# NUMPY MASTERY — CHAPTERS 12–14

```
NUMPY MASTERY
Current Chapters: 12 (Universal Functions / ufuncs), 13 (Mathematical Operations),
                  14 (Statistics with NumPy)
Progress:         14 / 75 chapters
Completed:        Ch1-11
Remaining:        61 chapters
```

With axis and broadcasting solid, this group is about the actual vocabulary of numerical
computation: ufuncs (the mechanism), aggregation functions (sum/mean/etc, built on axis), and
statistics (the ML-relevant layer on top).


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 12 — Universal Functions (ufuncs)

## Concept
A **ufunc** is a function that operates **element-wise** on ndarrays, implemented as a compiled
C loop, and supports broadcasting automatically. `np.add`, `np.multiply`, `np.sqrt`, `np.exp`
are all ufuncs — and so is the machinery *behind* `+`, `*`, etc. (`arr1 + arr2` literally calls
`np.add(arr1, arr2)` under the hood).

## Why it matters
Understanding ufuncs is what lets you predict: does this operation broadcast? Is it vectorized?
Can I fuse operations to avoid extra temporary arrays (`out=` parameter)?

## Industry use case
Applying an activation function (`np.exp`, `np.maximum` for ReLU) to an entire batch of neural
network activations in one call, image brightness adjustment (`np.clip`), feature transforms
(`np.log1p` for skewed distributions).


In [ ]:
a = np.array([1, 4, 9, 16])
b = np.array([1, 2, 3, 4])

print("np.add(a, b)      :", np.add(a, b), "   same as a + b:", np.array_equal(np.add(a, b), a + b))
print("np.sqrt(a)         :", np.sqrt(a))
print("np.exp(b)          :", np.exp(b))
print("np.log(a)          :", np.log(a))
print("np.maximum(a, 10)  :", np.maximum(a, 10))     # element-wise max -> ReLU-like clamping
print("np.clip(a, 2, 10)  :", np.clip(a, 2, 10))      # clamp into [2, 10]


### Expected Output
```
np.add(a, b)      : [ 2  6 12 20]    same as a + b: True
np.sqrt(a)         : [1. 2. 3. 4.]
np.exp(b)          : [ 2.71828183  7.3890561  20.08553692 54.59815003]
np.log(a)          : [0.         1.38629436 2.19722458 2.77258872]
np.maximum(a, 10)  : [10 10 10 16]
np.clip(a, 2, 10)  : [ 2  4  9 10]
```

### ufunc methods: `.reduce()`, `.accumulate()`, `.outer()`, `.at()`


In [ ]:
x = np.array([1, 2, 3, 4])

print("np.add.reduce(x)      :", np.add.reduce(x), "  (same as x.sum())")
print("np.multiply.reduce(x) :", np.multiply.reduce(x), "  (same as x.prod())")
print("np.add.accumulate(x)  :", np.add.accumulate(x), "  (running sum, like cumsum)")
print("np.multiply.outer([1,2],[10,20,30]):\n", np.multiply.outer([1, 2], [10, 20, 30]))

# .at() -- unbuffered in-place operation, handles REPEATED indices correctly
counts = np.zeros(5, dtype=int)
indices = np.array([0, 1, 1, 2, 2, 2])
np.add.at(counts, indices, 1)     # correctly increments repeated indices
print("np.add.at bincount-like:", counts)

# Contrast with naive fancy-indexing assignment, which does NOT accumulate repeats correctly:
wrong = np.zeros(5, dtype=int)
wrong[indices] += 1
print("naive wrong[indices]+=1 :", wrong, " (BUG: repeats got overwritten, not accumulated!)")


### Expected Output
```
np.add.reduce(x)      : 10   (same as x.sum())
np.multiply.reduce(x) : 24   (same as x.prod())
np.add.accumulate(x)  : [ 1  3  6 10]   (running sum, like cumsum)
np.multiply.outer([1,2],[10,20,30]):
 [[10 20 30]
  [20 40 60]]
np.add.at bincount-like: [1 2 3 0 0]
naive wrong[indices]+=1 : [1 1 1 0 0]  (BUG: repeats got overwritten, not accumulated!)
```

### The `+=` on fancy-indexed repeats trap — proven above, not hypothetical
Notice the two results **differ**: `np.add.at` correctly counts index `1` appearing twice
(→ `2`) and index `2` appearing three times (→ `3`); the naive `wrong[indices] += 1` gives `1`
for every touched position regardless of how many times it repeats. Why: `arr[idx] += 1` is
roughly sugar for `arr[idx] = arr[idx] + 1` — NumPy reads the **original** values at those
(possibly repeated) positions, adds 1 to each of the *read* copies, then scatter-writes the
results back; repeated indices in that write step just overwrite each other with the same
single-increment result rather than accumulating. `np.add.at(arr, indices, 1)` is the correct,
unbuffered way to accumulate into repeated indices — this is the exact mechanism behind
vectorized histogram/bincount-style code.

### Interview Insight
> "Why would you use `np.add.at()` instead of `arr[idx] += val`?"
Because `arr[idx] += val` does not correctly accumulate when `idx` has repeated values (it's
equivalent to a read-modify-write on the *original* array, so repeats overwrite rather than
accumulate); `np.add.at()` performs genuine, unbuffered, in-place accumulation, which matters
for tasks like building histograms or scatter-add operations (a core operation in some
deep-learning gradient computations).

### Best Practice: the `out=` parameter avoids temporary arrays
```python
result = np.empty_like(a)
np.multiply(a, 2, out=result)   # writes directly into `result`, no new array allocated
```
This matters in tight numerical loops / memory-constrained pipelines (Chapter 31).


---
# CHAPTER 13 — NumPy Mathematical Operations

## Concept
Aggregation functions (`sum`, `prod`, `mean`, `std`, `min`, `max`, `argmin`, `argmax`, ...)
reduce an array along one or more axes into fewer values, following exactly the axis reasoning
from Chapter 10.

## Why it matters
Nearly all EDA and preprocessing statistics are these functions applied with the right `axis`.


In [ ]:
X = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 100.]])   # note: outlier in last row

print("X:\n", X)
print()
print("X.sum()            :", X.sum(), "  (whole array, no axis)")
print("X.sum(axis=0)      :", X.sum(axis=0), "  (per-column sums)")
print("X.mean(axis=0)     :", X.mean(axis=0), "  (per-column means -- sensitive to the outlier!)")
print("X.min(axis=1)      :", X.min(axis=1))
print("X.max(axis=1)      :", X.max(axis=1))
print("X.argmax(axis=1)   :", X.argmax(axis=1), "  (INDEX of max per row, not the value)")
print("X.ptp(axis=0)      :", np.ptp(X, axis=0), "  (max - min per column, 'peak to peak')")


### Expected Output
```
X:
 [[  1.   2.   3.]
  [  4.   5.   6.]
  [  7.   8. 100.]]

X.sum()            : 136.0   (whole array, no axis)
X.sum(axis=0)      : [ 12.  15. 109.]   (per-column sums)
X.mean(axis=0)     : [ 4.  5. 36.33333333]   (per-column means -- sensitive to the outlier!)
X.min(axis=1)      : [1. 4. 7.]
X.max(axis=1)      : [ 3.  6. 100.]
X.argmax(axis=1)   : [2 2 2]   (INDEX of max per row, not the value)
X.ptp(axis=0)      : [ 6.  6. 97.]   (max - min per column, 'peak to peak')
```

### `argmax`/`argmin` — the function everyone forgets exists, then reinvents badly
`argmax` returns the **index** of the maximum, not the value — this is what you actually want
for "which class did the model predict" (`predictions.argmax(axis=1)` on a softmax output
matrix of shape `(n_samples, n_classes)` → shape `(n_samples,)` of predicted class indices).

### NaN-aware variants
```python
np.nanmean(x), np.nanstd(x), np.nanmin(x), np.nanmax(x), np.nanmedian(x)
```
These ignore `NaN` values instead of propagating them (`mean()` of an array containing any
`NaN` returns `NaN`). Essential for real-world data with missing values (Chapter 15).

### `where` parameter — conditional aggregation without a separate filter step
```python
X.mean(where=X > 0)   # mean of only the positive elements, without building an intermediate mask array
```

### Common Mistake
Using `.max()` when you meant `.argmax()` (or vice versa) — both compile and run without error,
they just answer different questions ("what is the biggest value" vs "where is the biggest
value"), and this is an extremely easy typo to make under interview/deadline pressure.

### Interview Insight
> "You have model output `probs` of shape `(n_samples, n_classes)`. How do you get predicted
> class labels?"
`probs.argmax(axis=1)` — collapse the class axis, keep the sample axis, get the **index** of
the highest probability per sample.


---
# CHAPTER 14 — Statistics with NumPy

## Concept
Beyond basic aggregation, NumPy provides percentile/quantile-based statistics, correlation, and
covariance — the toolkit for exploratory data analysis (EDA) and understanding relationships
between features.

## Why it matters
Outlier detection, feature scaling decisions, understanding which features are redundant
(highly correlated), and building the statistical intuition behind PCA (Chapter 42).


In [ ]:
rng = np.random.default_rng(42)
data = rng.normal(loc=50, scale=10, size=1000)   # simulate 1000 measurements, mean 50, std 10

print("mean               :", data.mean())
print("median             :", np.median(data))
print("std                :", data.std())
print("var                :", data.var())
print("25th percentile    :", np.percentile(data, 25))
print("75th percentile    :", np.percentile(data, 75))
print("0.25 quantile      :", np.quantile(data, 0.25), "  (quantile uses 0-1 scale, percentile uses 0-100)")
iqr = np.percentile(data, 75) - np.percentile(data, 25)
print("IQR                :", iqr)


### Expected Output (values will vary slightly by NumPy RNG version, seed is fixed so should match)
```
mean               : ~50.19
median             : ~49.9x
std                : ~9.9x
var                : ~9x^2
25th percentile    : ~43.x
75th percentile    : ~56.x
0.25 quantile      : ~43.x   (quantile uses 0-1 scale, percentile uses 0-100)
IQR                : ~13.x
```

### `np.percentile` vs `np.quantile`
Mathematically identical — `percentile(x, 75)` == `quantile(x, 0.75)`. The only difference is
the input scale (0–100 vs 0–1). Both support an `interpolation`/`method` parameter controlling
how to handle percentiles that fall between two data points (linear by default).

### IQR-based outlier detection (classic EDA technique, entirely NumPy)
```python
q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = data[(data < lower) | (data > upper)]
```

### Correlation and covariance


In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=200)
y = 2 * x + rng.normal(scale=0.5, size=200)   # y is linearly related to x, plus noise
z = rng.normal(size=200)                       # z is independent of x

stacked = np.vstack([x, y, z])   # shape (3, 200) -- np.corrcoef treats EACH ROW as a variable
corr = np.corrcoef(stacked)
cov = np.cov(stacked)

print("corrcoef matrix (3x3, rows=x,y,z):\n", np.round(corr, 2))
print()
print("cov matrix:\n", np.round(cov, 2))
print()
print("weighted average example:")
values = np.array([70, 80, 90])
weights = np.array([1, 2, 3])   # e.g. exam scores weighted by credit hours
print("np.average(values, weights=weights):", np.average(values, weights=weights))


### Expected Output (approximate, seeded)
```
corrcoef matrix (3x3, rows=x,y,z):
 [[ 1.    0.97 -0.0x]
  [ 0.97  1.   -0.0x]
  [-0.0x -0.0x  1.  ]]

cov matrix:
 [[ 1.03  2.06 -0.0x]
  [ 2.06  4.36 -0.0x]
  [-0.0x -0.0x  0.9x]]

weighted average example:
np.average(values, weights=weights): 83.33333333333333
```

### Reading the correlation matrix
`corr[i, j]` is the Pearson correlation between variable `i` and variable `j`. `x` and `y` show
strong positive correlation (~0.97, since `y` was constructed as `2x + noise`), while `x`/`z`
and `y`/`z` are near zero (independent by construction). The diagonal is always 1.0 (a variable
is perfectly correlated with itself).

### Common Mistake
Passing `np.corrcoef(X)` where `X` is your `(n_samples, n_features)` data matrix directly —
`corrcoef` treats **rows** as variables and **columns** as observations by default, the
opposite of the standard ML convention. For a typical `(n_samples, n_features)` matrix, you
need `np.corrcoef(X, rowvar=False)` to get the `(n_features, n_features)` correlation matrix
you almost certainly want.

### Interview Insight
> "You have a `(n_samples, n_features)` matrix and want the feature-feature correlation
> matrix. What's the pitfall?"
`np.corrcoef` defaults to `rowvar=True` (rows are variables) — you must pass `rowvar=False`
for a standard ML feature matrix, or transpose first. Forgetting this silently computes a
`(n_samples, n_samples)` sample-correlation matrix instead of the intended
`(n_features, n_features)` one — a shape that might even look "plausible" if n_samples is
small, making the bug easy to miss.


---
## Chapters 12–14 Summary

- **ufuncs** are the compiled, element-wise, broadcasting-aware engine behind every arithmetic
  operator; `.reduce()`/`.accumulate()`/`.outer()`/`.at()` expose that engine directly — `.at()`
  in particular is the correct tool for accumulating into repeated indices, where `+=` silently
  fails to accumulate.
- **Mathematical/aggregation operations** (`sum`, `mean`, `min`, `max`, `argmin`, `argmax`, ...)
  all follow the Chapter 10 axis-collapsing rule; `argmax`/`argmin` return **indices**, not
  values — critical for turning model probability outputs into predicted labels.
- **Statistics**: `percentile`/`quantile` (same math, different scale) power IQR-based outlier
  detection; `corrcoef`/`cov` reveal feature relationships but default to `rowvar=True`, which
  is usually **not** what you want for a standard `(samples, features)` ML matrix.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For `probs` of shape `(1000, 10)` (1000 samples, 10-class softmax
output), what is the shape of `probs.argmax(axis=1)`? What does each value represent?

**Exercise 2:** Given `X` of shape `(n_samples, n_features)`, write the one-line correct call
to get the `(n_features, n_features)` correlation matrix (mind the `rowvar` pitfall).

**Coding Challenge:** Implement `detect_outliers_iqr(x, k=1.5)` returning a boolean mask of
outliers using the IQR method, without loops. Then implement `robust_scale(x)` that subtracts
the median and divides by the IQR (a "robust" alternative to z-score standardization that's
less sensitive to outliers) — explain in a comment why this is preferable to mean/std scaling
for data like the outlier-containing `X` from the ufuncs section above.

**Debugging Challenge:** Reproduce the `np.add.at` example above with
`bins = np.array([3, 3, 3, 7, 7, 1])` instead of the notebook's `indices`. A teammate expected
`hist[3] == 3` and `hist[7] == 2` after `hist[bins] += 1`, but got `1` in both spots. Explain
exactly why (you already have the mechanism from the worked example), and give the one-line fix
using `np.add.at()`.

**Interview Questions:**
1. What is a ufunc, and how does `arr1 + arr2` relate to `np.add(arr1, arr2)`?
2. Why does `arr[idx] += 1` fail to correctly accumulate for repeated indices in `idx`?
3. What's the difference between `.max()` and `.argmax()`, and when would using the wrong one
   pass code review but produce a subtly wrong model?
4. What is the default `rowvar` behavior of `np.corrcoef`, and why is it a common source of
   bugs on standard ML feature matrices?

**Quiz:**
- Q1: True/False — `np.percentile(x, 50)` equals `np.median(x)`.
- Q2: Which function ignores NaN values when computing a mean?
- Q3: For model output `probs` of shape `(n, k)`, what does `probs.argmax(axis=1)` return —
  values or indices?

---


